In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import pysam
from Bio.Seq import Seq 
import copy
import warnings
warnings.filterwarnings('ignore')

In [33]:
import io
import itertools

from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
from alphagenome.visualization.plot_transcripts import TranscriptStyle
from IPython.display import clear_output
from alphagenome.data import track_data

In [34]:
def first_diff_index(s1, s2):
    for i, (a, b) in enumerate(zip(s1, s2)):
        if a != b:
            return i
    return None  # strings are identical

In [ ]:
df = pd.read_csv('./data/GATA1_oligo_synthesis_info.csv')
df = df[['WGATAR_id', 'type', 'oligo_230nt_chr','oligo_230nt_start', 'oligo_230nt_end',	'oligo_230nt_strand', 'oligo_seq_230nt']]
df = df.rename(columns={'oligo_230nt_chr':'chr', 'oligo_230nt_start':'start', 'oligo_230nt_end':'end', 'oligo_230nt_strand':'strand','oligo_seq_230nt':'seq'})
df = df.dropna(subset = ['WGATAR_id'])

In [36]:
from pyliftover import LiftOver
lo = LiftOver('hg19', 'hg38')

In [ ]:
GATA_id = df['WGATAR_id'].unique()
seq_dict = {}
for i in GATA_id: 
    chrom = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'chr'])[0]
    s = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'start'])[0]

    so = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'original'), 'seq'])[0]
    so = so.upper()

    sm = list(df.loc[(df['WGATAR_id'] == i) & (df['type'] == 'modified'), 'seq'])[0]
    sm = sm.upper()

    index = first_diff_index(so, sm)
    s = s + index
    s_38 = lo.convert_coordinate(chrom, s)[0][1]

    seq_dict[i] = copy.deepcopy([chrom, s_38])

In [ ]:
dna_model = dna_client.create('your_api_key')

sequence_length = "16KB"  # @param ["2KB", "16KB", "100KB", "500KB", "1MB"] { type:"string" }
sequence_length = dna_client.SUPPORTED_SEQUENCE_LENGTHS[f'SEQUENCE_LENGTH_{sequence_length}']

In [ ]:
track = ['WGATAR_id', 'EFO:0002067 ATAC-seq', 
         'EFO:0002067 TF ChIP-seq GATA1',
         'EFO:0002067 Histone ChIP-seq H3K27ac', 
         'EFO:0002067 TF ChIP-seq CTCF',
         'EFO:0002067 Histone ChIP-seq H3K27me3',
         'EFO:0002067 Histone ChIP-seq H3K36me3',
         'EFO:0002067 Histone ChIP-seq H3K4me1',
         'EFO:0002067 Histone ChIP-seq H3K4me2',
         'EFO:0002067 Histone ChIP-seq H3K4me3',
         'EFO:0002067 Histone ChIP-seq H3K79me2',
         'EFO:0002067 Histone ChIP-seq H3K9ac',
         'EFO:0002067 Histone ChIP-seq H4K20me1', 
         'EFO:0002067 total RNA-seq']

df_var = pd.DataFrame(columns=track)

In [ ]:
for k in seq_dict.keys():
    chrom, pos = seq_dict[k]
    variant = genome.Variant(
        chromosome=chrom,
        position=pos,
        reference_bases='A',
        alternate_bases='G',
    )
    interval = variant.reference_interval.resize(sequence_length)
    
    variant_scores = dna_model.score_variant(
        interval=interval,
        variant=variant,
        variant_scorers=list(variant_scorers.RECOMMENDED_VARIANT_SCORERS.values()),)

    df_scores = variant_scorers.tidy_scores(variant_scores)
    df_scores = df_scores[df_scores['biosample_name'] == 'K562']

    var_list = [k]
    for t in track[1:]:
        v = df_scores[df_scores['track_name'] == t]['quantile_score'].mean()
        var_list.append(copy.deepcopy(v))

    df_var.loc[len(df_var)] = copy.deepcopy(var_list)

In [ ]:
df_var.to_csv('./data/var_scores.csv', index=None)